In [14]:
import pandas as pd
import unicodedata
from pathlib import Path

def normalize(text):
    text = unicodedata.normalize("NFKD", str(text))
    text = text.encode("ASCII", "ignore").decode("utf-8")
    return text.upper().strip()

DATA_DIR = Path("../data").resolve()
input_path = DATA_DIR / "input/ibge/population/tabela4714.csv"

# Pular as 4 primeiras linhas
df = pd.read_csv(
    input_path,
    sep=",",
    skiprows=4,
    encoding="utf-8",
    engine="python"
)


df.columns = ["nivel", "cd_mun", "municipio", "populacao", "unidade"]

# Manter apenas municípios
df = df[df["nivel"] == "MU"]

# Separar cidade e UF
df[["nm_mun", "uf_raw"]] = df["municipio"].str.extract(r"^(.*)\s\((.*)\)$")

# Normalizar nome
df["nm_mun"] = df["nm_mun"].apply(normalize)

# Sigla UF já vem correta
df["sigla_uf"] = df["uf_raw"]

df["populacao"] = pd.to_numeric(df["populacao"], errors="coerce").astype("Int64")


# Selecionar final
df_final = df[["cd_mun", "nm_mun", "sigla_uf", "populacao"]]

print(len(df_final))  # deve dar ~5570





5570


In [15]:
#output_csv = DATA_DIR / "output/br_population_municipio_2022.csv"
#df_final.to_csv(output_csv, index=False)
#print("CSV salvo em:", output_csv)


output_parquet = DATA_DIR / "output/br_population_municipio_2022.parquet"

df_final.to_parquet(output_parquet, index=False)

print("Parquet salvo em:", output_parquet)


Parquet salvo em: G:\My Drive\Github\py-2026-map-shaper\data\output\br_population_municipio_2022.parquet


In [16]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/output")
DATA_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_PATH = DATA_DIR / "br_population_municipio_2022.parquet"

import pandas as pd

df = pd.read_parquet(PARQUET_PATH)

import os
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
import json
import time

load_dotenv(find_dotenv())
DB_URL = os.environ.get("DB_URL")
if not DB_URL:
    raise EnvironmentError("DB_URL environment variable is not set.")

engine = create_engine(DB_URL)

print("Enviando para Postgres...")
df.to_sql(
    "ibge_tabela4714_br_population_municipio_2022",
    engine,
    schema="staging",
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=2000
)

print("Enviando para Postgres...")

end_time = time.time()

print(f"Enviado para Postgres. Tempo total: {(end_time - start_time) / 60:.2f} minutos")

Enviando para Postgres...
Enviando para Postgres...
Enviado para Postgres. Tempo total: 0.15 minutos
